# BMBT tier-3: Banglish transliteration model (from scratch)

Trains the tier-3 model for the Banglish pipeline (`bntok.banglish`), on Colab's free T4 tier.
No pretrained checkpoint anywhere - random init, trained here.

Checkpoints go to your mounted Drive, so this survives Colab's ~90min-idle / ~12hr session limits:
just re-run this notebook and training resumes from the last checkpoint automatically.

Before running: set `DRIVE_DATA_DIR` and `DRIVE_CKPT_DIR` below to folders in your own Drive.
Upload `artifacts/banglish-translit-data/{train.tsv,dev.tsv,vocab.json}` (from the repo, already
assembled by `scripts/assemble_banglish_translit_dataset.py`) into `DRIVE_DATA_DIR` first.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
DRIVE_DATA_DIR = '/content/drive/MyDrive/banglish-translit-data'
DRIVE_CKPT_DIR = '/content/drive/MyDrive/banglish-translit-ckpt'

import os
assert os.path.exists(os.path.join(DRIVE_DATA_DIR, 'train.tsv')), (
    f'Upload train.tsv/dev.tsv/vocab.json to {DRIVE_DATA_DIR} first '
    '(from artifacts/banglish-translit-data/ in the repo).'
)

In [ ]:
import os
os.chdir('/content')
if os.path.exists('/content/bornomala'):
    !rm -rf /content/bornomala
!git clone --depth 1 https://github.com/konkomaji/bornomala.git /content/bornomala
os.chdir('/content/bornomala/bengali-tokenizer')
!pip install -e . -q

## Train

`--max-steps` and `--batch-size` are conservative defaults for a T4; raise `--batch-size` if you
have headroom (check `nvidia-smi`). Re-running this cell (or the whole notebook) resumes from the
latest checkpoint in `DRIVE_CKPT_DIR` automatically - no flag needed.

In [ ]:
!python scripts/train_banglish_translit.py \
  --data-dir "{DRIVE_DATA_DIR}" \
  --ckpt-dir "{DRIVE_CKPT_DIR}" \
  --device cuda \
  --batch-size 256 \
  --max-steps 20000 \
  --save-every 500 \
  --eval-every 500 \
  --log-every 50

## Evaluate

Against Dakshina's reserved lexicon TEST split - real held-out data, never touched by training or
the tier-1 lookup table. Two numbers: strict word-level exact-match, and character error rate
(partial credit for close-but-wrong). Needs `dakshina_dataset_v1.0/bn` uploaded to Drive too, or
downloaded fresh here (official source, verified reachable - see docs/known-issues.md).

First result measured (greedy decoding, 20k-step small model): **45.1% exact-match, 16.9% CER.**
Two ways to push it, cheapest first:
1. **Beam search** - zero retraining cost, just changes how the existing checkpoint is queried.
   Try this first (next cell, `--beam-size 5`).
2. **Bigger model / longer training** - a real retrain, more GPU time. Further down, its own
   section, only worth running once beam search's ceiling on the current checkpoint is known.

In [ ]:
import os
DAKSHINA_DIR = '/content/dakshina_dataset_v1.0/bn'
if not os.path.exists(DAKSHINA_DIR):
    !curl -s -o /content/dakshina_v1.0.tar https://storage.googleapis.com/gresearch/dakshina/dakshina_dataset_v1.0.tar
    !tar -xf /content/dakshina_v1.0.tar -C /content dakshina_dataset_v1.0/bn/lexicons
    !rm /content/dakshina_v1.0.tar

In [ ]:
import glob
latest_ckpt = sorted(
    glob.glob(os.path.join(DRIVE_CKPT_DIR, 'step-*.pt')),
    key=lambda p: int(p.split('step-')[-1].split('.pt')[0]),
)[-1]
print('evaluating', latest_ckpt, '- beam search, beam size 5')

!python scripts/eval_banglish_translit.py \
  --dakshina-dir "{DAKSHINA_DIR}" \
  --data-dir "{DRIVE_DATA_DIR}" \
  --ckpt "{latest_ckpt}" \
  --device cuda \
  --beam-size 5

## Optional: bigger model, longer training

A real retrain, not free like beam search - only run this after seeing what beam search alone
gets on the existing checkpoint above. Bumps capacity (d_model 256->384, layers 4->6, feedforward
1024->1536 - same scale-up as `colab/train_bmbt_downstream_eval.ipynb`'s own LM) and training
length (20k->30k steps). Uses a SEPARATE `DRIVE_CKPT_DIR_V2`: the architecture differs from the
first run's checkpoints, so it cannot resume from or share a directory with them.

In [ ]:
DRIVE_CKPT_DIR_V2 = '/content/drive/MyDrive/banglish-translit-ckpt-v2'

!python scripts/train_banglish_translit.py \
  --data-dir "{DRIVE_DATA_DIR}" \
  --ckpt-dir "{DRIVE_CKPT_DIR_V2}" \
  --device cuda \
  --batch-size 256 \
  --d-model 384 \
  --nhead 8 \
  --num-layers 6 \
  --dim-ff 1536 \
  --max-steps 30000 \
  --save-every 500 \
  --eval-every 500 \
  --log-every 50

In [ ]:
latest_ckpt_v2 = sorted(
    glob.glob(os.path.join(DRIVE_CKPT_DIR_V2, 'step-*.pt')),
    key=lambda p: int(p.split('step-')[-1].split('.pt')[0]),
)[-1]
print('evaluating', latest_ckpt_v2, '- beam search, beam size 5')

!python scripts/eval_banglish_translit.py \
  --dakshina-dir "{DAKSHINA_DIR}" \
  --data-dir "{DRIVE_DATA_DIR}" \
  --ckpt "{latest_ckpt_v2}" \
  --device cuda \
  --beam-size 5

## Bring the checkpoint home

Whichever checkpoint scored best above - `DRIVE_CKPT_DIR` (small) or `DRIVE_CKPT_DIR_V2`
(bigger) - download its `step-<N>.pt` and `vocab.json`, drop them into
`artifacts/banglish-translit-model/` in the repo, and wire the trained model into
`bntok.banglish.transliterate()`'s `tier3_fn` slot (the hook already exists and is tested - see
`tests/test_banglish.py`). Also note in the commit which decoder (greedy or beam, and beam size)
the reported numbers used - inference needs to match at serving time too, not just at eval time.